# 
*Column Transformer*

In [1]:
import numpy as np 
import pandas as pd

In [2]:
df = pd.read_csv("/home/raghav/ML(100dayseries)/100-days-of-ml-campusx/feature_engineering/feature transformation/covid_toy.csv")
df.sample(7)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
70,68,Female,101.0,Strong,Delhi,No
62,56,Female,104.0,Strong,Bangalore,Yes
3,31,Female,98.0,Mild,Kolkata,No
17,40,Female,98.0,Strong,Delhi,No
72,83,Female,101.0,Mild,Kolkata,No
16,69,Female,103.0,Mild,Kolkata,Yes


*gender* and *city* is nominal data

*cough* is ordinal data

In [6]:
df['cough'].value_counts()#Ordinal encoder

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [7]:
df['city'].value_counts()#one hot encoder

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [13]:
df.isnull().sum()# has null values

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

## *step 1 : train_test_split*

In [75]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split( df.drop(columns=['has_covid']),
                                                     df['has_covid'],
                                                     test_size=0.2,
                                                     random_state=0)

In [20]:
x_train.shape

(80, 5)

### filling fever column missing values

In [29]:
from sklearn.impute import SimpleImputer

si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])

x_test_fever = si.fit_transform(x_test[['fever']])

x_train_fever.shape

(80, 1)

## *Ordinal encoder : encoding cough column*

In [40]:
from sklearn.preprocessing import OrdinalEncoder
oe=OrdinalEncoder(categories=[['Mild','Strong']])#MULTIPLE MAKE LIST IN LIST

x_train_cough=oe.fit_transform(x_train[['cough']])
x_test_cough=oe.fit_transform(x_test[['cough']])

x_train_cough.shape

(80, 1)

In [49]:
from sklearn.preprocessing import OneHotEncoder

ohe=OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender', 'city']])

x_test_gender_city = ohe.fit_transform(x_test[['gender', 'city']])
x_train_gender_city

numpy.ndarray

In [57]:
x_train_age=x_train.drop(columns=['gender','cough','city','fever']).values
x_test_age=x_test.drop(columns=['gender','cough','city','fever']).values
x_train_age.shape

(80, 1)

In [64]:
x_train_transformed = np.concatenate((x_train_age, x_train_fever, x_train_gender_city, x_train_cough), axis = 1)
x_test_transformed = np.concatenate((x_test_age, x_test_fever, x_test_gender_city, x_test_cough), axis = 1)
x_train_transformed.shape

(80, 7)

# *Using column transformer*

In [77]:
from sklearn.compose  import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

transformer = ColumnTransformer( transformers = [
                                                ('tf1',SimpleImputer(),['fever']),
                                                ('tf2',OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
                                                ('tf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city']),
                                                ],remainder = 'passthrough')#drop or passthrough rest columns

In [83]:
x_train_trans =transformer.fit_transform(x_train)
x_test_trans =transformer.fit_transform(x_test)
le=LabelEncoder()

le.fit(y_train)
y_train_trans = le.transform(y_train)
y_test  = le.transform(y_test)

In [87]:
x_train_trans.shape

(80, 7)

In [89]:
y_train_trans

array([1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0,
       1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0])